In [28]:
import pandas as pd

In [29]:
all_2247 = pd.read_csv("../Data/ST-2247.csv")
all_2252 = pd.read_csv("../Data/ST-2252.csv")
all_2425 = pd.read_csv("../Data/ST-2425.csv")

---
#### 주중 주말 매핑

In [30]:
import pandas as pd

dfs = [all_2247, all_2252, all_2425]

# 주말로 강제 지정할 날짜 리스트 (YYYYMMDD)
holiday_dates = [
    20240101, 20240209, 20240210, 20240211, 20240212,
    20240301, 20240410, 20240505, 20240506, 20240515,
    20240606, 20240815, 20240916, 20240917, 20240918,
    20241003, 20241009, 20241225
]
holiday_dates = set(holiday_dates)

# 요일 매핑
weekday_map = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}

for df in dfs:
    # 이용시각이 문자열일 수도 있으니 datetime 변환
    df["이용시각"] = pd.to_datetime(df["이용시각"])
    
    # 요일 컬럼
    df["요일"] = df["이용시각"].dt.weekday.map(weekday_map)
    
    # 주말 컬럼
    df["주말"] = df["이용시각"].dt.weekday.apply(lambda x: "주말" if x >= 5 else "주중")
    
    # YYYYMMDD 숫자 형태로 변환해서 휴일 강제 주말 처리
    yyyymmdd = df["이용시각"].dt.strftime("%Y%m%d").astype(int)
    df.loc[yyyymmdd.isin(holiday_dates), "주말"] = "주말"


In [31]:
all_2247.to_csv("../Data/ST-2247.csv")
all_2252.to_csv("../Data/ST-2252.csv")
all_2425.to_csv("../Data/ST-2425.csv")

----
#### 날씨 매핑

In [32]:
import requests
import pandas as pd

# 은평구청 좌표 (대표 지점)
lat, lon = 37.60279, 126.92918

# 매핑 대상 데이터프레임들
dfs = [all_2247, all_2252, all_2425]

# 이용시각 전체 범위 계산 (Open-Meteo는 날짜 단위 범위를 받음)
for df in dfs:
    df["이용시각"] = pd.to_datetime(df["이용시각"])

start_dt = min(df["이용시각"].min() for df in dfs).floor("D")
end_dt = max(df["이용시각"].max() for df in dfs).ceil("D")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_dt.date().isoformat(),
    "end_date": end_dt.date().isoformat(),
    "hourly": [
        "temperature_2m",
        "precipitation",
        "snowfall",
        "windspeed_10m"
    ],
    "timezone": "Asia/Seoul"
}

res = requests.get(url, params=params).json()
weather = pd.DataFrame(res["hourly"])

# 컬럼명 한국어로 변경
weather = weather.rename(columns={
    "temperature_2m": "기온",
    "precipitation": "강수량",
    "snowfall": "적설량",
    "windspeed_10m": "풍속"
})

# 시간 타입 맞추기
weather["time"] = pd.to_datetime(weather["time"]).dt.tz_localize(None)

print("weather 범위:", weather["time"].min(), "~", weather["time"].max())

# 이용시각 기준으로 시간 단위 매핑
for i, df in enumerate(dfs):
    df["이용시각"] = pd.to_datetime(df["이용시각"])
    # 이용시각이 타임존 포함이면 KST로 맞춘 뒤 naive로 변환
    if df["이용시각"].dt.tz is not None:
        df["날씨시각"] = (
            df["이용시각"].dt.tz_convert("Asia/Seoul").dt.tz_localize(None).dt.floor("h")
        )
    else:
        df["날씨시각"] = df["이용시각"].dt.floor("h")

    print("df 이용시각 범위:", df["이용시각"].min(), "~", df["이용시각"].max())
    print("df 날씨시각 범위:", df["날씨시각"].min(), "~", df["날씨시각"].max())

    df = df.merge(
        weather,
        left_on="날씨시각",
        right_on="time",
        how="left"
    )

    df.drop(columns=["time"], inplace=True, errors="ignore")
    dfs[i] = df

    print(f"매칭률: {df['기온'].notna().mean():.1%}")


# 원본 변수에 반영
all_2247, all_2252, all_2425 = dfs


weather 범위: 2024-01-01 00:00:00 ~ 2025-01-01 23:00:00
df 이용시각 범위: 2024-01-01 00:05:00 ~ 2024-12-31 22:55:00
df 날씨시각 범위: 2024-01-01 00:00:00 ~ 2024-12-31 22:00:00
매칭률: 100.0%
df 이용시각 범위: 2024-01-01 04:35:00 ~ 2024-12-31 23:12:00
df 날씨시각 범위: 2024-01-01 04:00:00 ~ 2024-12-31 23:00:00
매칭률: 100.0%
df 이용시각 범위: 2024-01-01 01:40:00 ~ 2024-12-31 22:45:00
df 날씨시각 범위: 2024-01-01 01:00:00 ~ 2024-12-31 22:00:00
매칭률: 100.0%


In [33]:
all_2247

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,기준_날짜,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,...,경도,이용기준,이용시각,요일,주말,날씨시각,기온,강수량,적설량,풍속
0,0,0,0,130,20240101,ST-1036,역촌동_001_1,ST-2247,신사2동_013_1,1.0,...,126.916794,도착시간,2024-01-01 16:40:00,월,주말,2024-01-01 16:00:00,5.9,0.0,0.0,10.8
1,1,1,1,177,20240101,ST-463,증산동_004_1,ST-2247,신사2동_013_1,1.0,...,126.909897,도착시간,2024-01-01 19:13:00,월,주말,2024-01-01 19:00:00,0.2,0.0,0.0,8.0
2,2,2,2,235,20240101,ST-2247,신사2동_013_1,ST-2247,신사2동_013_1,1.0,...,126.906082,도착시간,2024-01-01 18:16:00,월,주말,2024-01-01 18:00:00,2.0,0.0,0.0,7.6
3,3,3,3,456,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,...,126.916985,도착시간,2024-01-01 22:30:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
4,4,4,4,661,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,...,126.916985,도착시간,2024-01-01 22:49:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25708,25708,25708,25708,1997775,20241231,ST-463,증산동_011_1,ST-2247,신사2동_013_1,1.0,...,126.909897,도착시간,2024-12-31 14:33:00,화,주중,2024-12-31 14:00:00,1.8,0.0,0.0,12.5
25709,25709,25709,25709,1997782,20241231,ST-2247,신사2동_013_1,ST-2537,상암동_029_1,1.0,...,126.906082,출발시간,2024-12-31 15:30:00,화,주중,2024-12-31 15:00:00,1.8,0.0,0.0,12.4
25710,25710,25710,25710,1997885,20241231,ST-2247,신사2동_013_1,ST-2240,이촌2동_012_1,1.0,...,126.906082,출발시간,2024-12-31 10:00:00,화,주중,2024-12-31 10:00:00,-0.7,0.0,0.0,12.2
25711,25711,25711,25711,1997897,20241231,ST-2825,응암2동_050_1,ST-2247,신사2동_013_1,1.0,...,126.919624,도착시간,2024-12-31 07:52:00,화,주중,2024-12-31 07:00:00,-3.5,0.0,0.0,13.0


In [34]:
all_2247.to_csv("../Data/ST-2247.csv")
all_2252.to_csv("../Data/ST-2252.csv")
all_2425.to_csv("../Data/ST-2425.csv")